In [ ]:
#!git clone https://github.com/nlacslab/kaznlp.git
#!mkdir /content/temp
#!mv /content/kaznlp/kaznlp /content/temp/kaznlp
#!rm -rf /content/kaznlp
#!mv /content/temp/kaznlp /content/kaznlp

In [ ]:
!pip install datasketch
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.9 MB/s eta 0:00:00


In [ ]:
import textstat

UNWANTED_PHRASES = [
    "Әлеуметтік желілерде бөлісіңіз",
    "ВКонтакте бөлу",
    "ВКонтакте бөлісу",
    "Facebookта бөлісу",
    "Төменде пікір қалдырыңыз",
    "Пікір қалдыру",
    "Пікір",
    "пікір",
]

def clean_text(txt: str) -> str:
    for phrase in UNWANTED_PHRASES:
        txt = txt.replace(phrase, "")
    txt = txt.strip()

    if len(txt) < 10:
        return ""
    return txt


def get_word_count(text: str) -> int:
    return len(text.split())


def get_seed_first_sentence(text: str) -> str:
    parts = re.split(r'(?<=[.!?])\s+', text)
    for part in parts:
        s = part.strip()
        if len(s) > 10:
            return s
    return text[:200].strip()


def format_data(id, title, url, text, title_kaz = ""):
  return {
      'id': id,
      'title': title.lower(),
      'title_kaz': title_kaz.lower(),
      'url': url,
      'age_group': None,
      'theme': None,
      'length': get_word_count(text),
      'personalization': {'child_name': None, 'parent_role': None, 'location': None},
      'seed': get_seed_first_sentence(text),
      'planner': None,
      'story': text
  }

# Get Data

## Scraping stories from ertegiler.kz

In [ ]:
stories = []

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

pattern = re.compile(r"https://ertegiler\.kz/story/[\w\-\d\,]+", re.IGNORECASE)
stories_ertegiler = set()

for _ in range(1, 26+1):
  url = f"https://ertegiler.kz/?page={_}"
  response = requests.get(url)
  html_content = response.text
  stories_ertegiler |= set(pattern.findall(html_content))

In [ ]:
from urllib.parse import urlparse

stories = [
  format_data(
      id=f'ertegiler_{i:04d}',
      title=urlparse(url).path.rsplit("/", 1)[-1].replace("-", " ").replace("_"," "),
      url=url,
      text=" ".join([p.get_text(strip=True) for p in BeautifulSoup(requests.get(url).text, 'html.parser').find('div', itemprop='articleBody').find_all('p')]),
  )
  for i, url in enumerate(stories_ertegiler)
]

In [ ]:
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests

def extract_story(url):
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    h1 = soup.find("h1")
    if h1:
        real_title = h1.get_text(strip=True)
    else:
        real_title = urlparse(url).path.rsplit("/", 1)[-1].replace("-", " ")

    body = soup.find("div", itemprop="articleBody")

    if body:
        paragraphs = [p.get_text(" ", strip=True) for p in body.find_all("p")]
        story_text = " ".join(paragraphs)
    else:
        story_text = ""

    return real_title, story_text


stories = []
for i, url in enumerate(stories_ertegiler):
    title, text = extract_story(url)

    stories.append(
        format_data(
            id=f'ertegiler_{i:04d}',
            title=urlparse(url).path.rsplit("/", 1)[-1].replace("-", " ").replace("_"," "),
            url=url,
            text=text,
            title_kaz=title
        )
    )


In [ ]:
import json
file_path = "/content/ertegiler_kz.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

## Scraping stories from balalaralemi.kz

In [ ]:
stories = []

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

pattern = re.compile(r"/article/\d+/[\w\-\d,]+", re.IGNORECASE)
stories_balalaralemi = set()
for _ in range(12):
  url = f"https://balalaralemi.kz/catalog/view/1?page={_}"
  response = requests.get(url)
  html_content = response.text
  #print(html_content)
  stories_balalaralemi |= set(pattern.findall(html_content))

In [ ]:
import requests
for i, article in enumerate(stories_balalaralemi):
  url = f"https://balalaralemi.kz{article}"

  html = requests.get(url).text
  soup = BeautifulSoup(html, "html.parser")

  title_top = soup.find('div', class_='title_top')
  if title_top and title_top.find('h1'):
      title_kaz = title_top.find('h1').get_text(strip=True)
  else:
      title_kaz = None
  print(title_kaz)
  content_div = soup.find('div', class_='content')
  for tag in content_div.find_all(['script', 'style', 'ul', 'div', 'img']):
    tag.decompose()
  text = content_div.get_text(separator='\n', strip=True)


  if not text:
    continue
  json_format = format_data(id=f'balalaralemi_{i:04d}', title=article.split('/')[-1], url=url, text=text, title_kaz=title_kaz)

  stories.append(json_format)

Кішкентай тас пен құмырсқа
Тапқыр қоян
Жақсылық пен жамандық
Ақылды егінші
Лақтың мүйізі
Қасық пен Айыр
Қиялшылдар
Қасқыр мен егінші
Түлкі мен Бөдене
Қаңбақ шал
Уәзір мен шал
Арыстан мен түлкі
Патша мен құмырсқа
Ұр,тоқпақ!
Патшаның қызы неге бақытсыз
Түбір мен Қосымша
Түлкі, аю және қойшы
Бере білген  бөле білер
Қоян мен тасбақа
Етікші
Кішкентай маса
Сәбиге өмір сыйлаған құдырет
Үш аю
Қорқақ қоян
Балықтар неге сөйлемейді?
Еріншек
Бір үзім нан
Ақшақар және жеті гном
Тышқан мен нәресте
Қорқақ қоянның жауабы
Әнші қоян
Қардан жасалған гүл
Жеті өнерпаз
Қарттың ұлына өсиеті
Екі дос пен аю
Соқыр күйші
Өнерпаз Патшайым
Гүл
Алтын балық
Ақылды етікші
Қарт пен тапқыр жігіт
Алтын, күміс
Ғылманның өнері
Қу түлкі
Ай мен күн
Қасқыр мен түлкі
Сөйлей алатын құс
Кім күшті?
Түлкі мен ешкі
Алтын санаған аштан өледі
Өнеге
Тазша бала
Алып қара құс - Самұрық
Түлкі,тасбақа және кене
Акылды қоян
Ай неге жалаңаш қалды
Өгіз
Алтын балта
Арыстан мен тышқан
Түлкі мен Құмыра
Қасқыр мен кісі
Еңбек етсең ерінбей...
Бе

In [ ]:
import json
file_path = "/content/balalaralemi_kz.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

## Scraping stories from bilim-all.kz

In [ ]:
stories = []

In [ ]:
import time
pattern = re.compile(r"/article/\d+-[\w-]+", re.IGNORECASE)
stories_bilim_al = set()
for _ in range(30):
  time.sleep(3)
  url = f"https://bilim-all.kz/article/list?id=4&page={_}"
  response = requests.get(url)
  html_content = response.text
  #print(html_content)
  stories_bilim_al |= set(pattern.findall(html_content))

In [ ]:
BASE_URL = "https://bilim-all.kz"

n = 0

for idx, article in enumerate(stories_bilim_al):
    time.sleep(3)

    url = f"{BASE_URL}{article}"
    #print(f"\n[{idx}] Downloading:", url)

    html_content = requests.get(url).text
    soup = BeautifulSoup(html_content, "html.parser")

    # ---- TITLE ----
    h1 = soup.find("h1")
    if not h1:
        continue
    title = h1.get_text(strip=True)
    print("TITLE:", title)

    # ---- STORY TEXT ----
    content_div = soup.find("div", class_="blogtext")
    story_paragraphs = []

    if content_div:
        for p in content_div.find_all("p"):
            txt = clean_text(p.get_text(" ", strip=True))
            if txt:
                story_paragraphs.append(txt)

    story_text = "\n".join(story_paragraphs).strip()

    if not story_text:
        continue

    json_format = format_data(id=f"bilim_{idx:04d}", title=title, url=url, text=story_text, title_kaz=title)

    stories.append(json_format)

    n += 1


TITLE: Қызыл қораз
TITLE: Екі суайт
TITLE: Ең керемет дәм туралы
TITLE: Мейірімді аяз жайлы ертегі
TITLE: Сандық самолет
TITLE: Өкпешіл көжек
TITLE: Адам болған жылан
TITLE: Тышқан мен жылан
TITLE: Қыз бен тазша
TITLE: Жезтырнақ
TITLE: Ермек
TITLE: Рахымның бай болмағаны
TITLE: Алдар Көсе мен Алаша хан
TITLE: Ахметбек пен Хасен
TITLE: Ай мен Күн
TITLE: Түлкі мен бөдене
TITLE: Шәкірат пен Шәкір
TITLE: Ақылды күшік
TITLE: Тұз басты жыланның хикаясы
TITLE: Су асты елінің ханзадасы мен ханшайым
TITLE: Дана қораз
TITLE: Аюбала
TITLE: Алтын пышақ
TITLE: Аю мен бал арасы
TITLE: Диірмендегі кедей жұмысшы мен мысық
TITLE: Алтын ат
TITLE: Иттің сырттаны
TITLE: Алдар Көсе (II нұсқа)
TITLE: Кім күшті? / Дүниеде не күшті? (ертегі)
TITLE: Хамелеонның түсі қандай?
TITLE: Қазақстан Барысы
TITLE: Қысқы ертегі
TITLE: Үш ағайынды жігіт
TITLE: Былтыр ұрды
TITLE: Түс сатқан тазша
TITLE: Кірпі мен көртышқан
TITLE: Сиқыршының сыйы осы
TITLE: Қардан жасалған гүл
TITLE: Түс көрген патша
TITLE: Сабан, көмір жән

In [ ]:
import json
file_path = "/content/bilim_all_kz.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

## Getting stories written by Ybyrai Altynsarin from wikisource

In [ ]:
stories = []

In [ ]:
import time
import urllib.parse

API = "https://wikisource.org/w/api.php"
HEADERS = {"User-Agent": "ErtegilerimAI/0.1"}

# 1) category → stories
params = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Category:Ыбырай_Алтынсариннің_әңгімелері",
    "cmlimit": "max",
    "format": "json"
}

data = requests.get(API, params=params, headers=HEADERS).json()
pages = data["query"]["categorymembers"]

# 2) load each story
for i, page in enumerate(pages):
    title = page["title"]
    print("Loading:", title)

    params = {
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "formatversion": "2"
    }

    r = requests.get(API, params=params, headers=HEADERS)
    html = r.json()["parse"]["text"]

    soup = BeautifulSoup(html, "html.parser")
    text = "\n".join([p.get_text(" ", strip=True) for p in soup.find_all("p")])

    json_format = format_data(id=f"Altynsarin_{i:04d}", title=title, url=f"https://wikisource.org/wiki/{urllib.parse.quote(title)}", text=text, title_kaz=title)
    stories.append(json_format)

    time.sleep(0.5)




Loading: Ізбасты
Loading: Абылай хан
Loading: Айуанның естесі көп, бірақ адамдай толық ақылы жоқ
Loading: Алтын шеттеуік
Loading: Асыл шөп
Loading: Аурудан аяған күштірек
Loading: Ақымақ дос
Loading: Білгеннің пайдасы
Loading: Бір уыс мақта
Loading: Бай мен жарлы баласы
Loading: Байұлы
Loading: Баланың айласы
Loading: Бақша ағаштары
Loading: Дүние қалай етсең табылады
Loading: Жаман жолдас
Loading: Жамандыққа жақсылық
Loading: Жан-жануарлардың дауласқаны
Loading: Жомарт
Loading: Жәнібек батыр
Loading: Зеректік
Loading: Лұқпан әкім
Loading: Малды пайдаға жарату
Loading: Мейірімді бала
Loading: Мұжық пен жасауыл
Loading: Мұңсыз адам
Loading: Надандық
Loading: Петр патшаның тергелгені
Loading: Полкан деген ит
Loading: Салақтық
Loading: Сараңдық пен жинақтылық
Loading: Сауысқан мен қарға
Loading: Сақып
Loading: Силинші деген ханым
Loading: Сәтемір хан
Loading: Таза бұлақ
Loading: Талаптың пайдасы
Loading: Тышқанның өсиеті
Loading: Түлкі мен ешкі
Loading: Шеше мен бала
Loading: Қанағат
Load

In [ ]:
import json
file_path = "/content/wiki_source_ybyrai_altynsarin.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

#Checkpoint

In [ ]:
import json

def load_jsons(*args):
  out = []
  for file in args:
    with open(file, "r", encoding="utf-8") as json_file:
      print(file)
      out += json.load(json_file)
  return out

In [ ]:
import json

stories = load_jsons(
    "/content/wiki_source_ybyrai_altynsarin.json",
    #"/content/bilim_all_kz.json",
    "/content/balalaralemi_kz.json",
    "/content/ertegiler_kz.json"
    )

file_path = "/content/raw_data.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

/content/wiki_source_ybyrai_altynsarin.json
/content/balalaralemi_kz.json
/content/ertegiler_kz.json


In [ ]:
import json
with open(file_path, "r", encoding="utf-8") as json_file:
     stories = json.load(json_file)
print(len(stories))

306


# Clean Data

removed stories with bad words and empty stories

In [ ]:
bad_words = ["сүйіс", "емшек"]

stories = [
    story
    for story in stories
    if len(story["story"]) != 0 and not any(bad in story["story"].lower() for bad in bad_words)
]

# Minhash (Remove stories that are extemly similat

In [ ]:
#MinHash to remove
from datasketch import MinHash
from itertools import combinations
from random import choice
import re

similarity_bound = 0.9
stories_minhash = {}

old_stories = len(stories)


for k, v in enumerate(stories):
  mhash = MinHash(num_perm=256)
  for token in re.split(r"[ ,\.!?]+", v["story"]):
    mhash.update(token.encode('utf-8'))
  stories_minhash[k] = mhash


to_remove = set(
    choice([k1, k2])
    for (k1, m1), (k2, m2) in combinations(stories_minhash.items(), 2)
    if m1.jaccard(m2) > similarity_bound
)

print(f"Removing {len(to_remove)} stories")

stories = [story for i, story in enumerate(stories) if i not in to_remove]

print(f"{len(stories)}/{old_stories} stories left")

Removing 3 stories
291/294 stories left


# Turn data to JSON

In [ ]:
import json

file_path = "/content/scraped_stories.json"
with open(file_path, "w") as json_file:
  json.dump(stories, json_file, indent=4)

In [ ]:
import json
file_path = "/content/scraped_stories.json"
with open(file_path, "r", encoding="utf-8") as json_file:
    doc = json.load(json_file)

Add labels


In [ ]:
import json
import re
import numpy as np

input_file = "scraped_stories.json"
output_json = "/content/scraped_stories_labeled.json"
output_train = "/content/train_with_tags.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    stories = json.load(f)

print(f"Loaded {len(stories)} stories.")


# keywords for Kazakh context, need to be checked mannually.
THEME_KEYWORDS = {
    "friendship": ["дос", "достық", "жолдас", "көмек", "бірлік"],
    "bravery": ["батыр", "ерлік", "қаһарман", "күшті", "жеңді", "қорғау"],
    "family": ["ана", "әке", "әже", "ата", "бауыр", "бала", "ұл", "қыз"],
    "nature": ["орман", "дала", "тау", "өзен", "көл", "ағаш", "гүл", "жел", "жаңбыр"],
    "animals": ["қасқыр", "түлкі", "қоян", "аю", "арыстан", "жолбарыс", "құс", "ат", "жылқы"],
    "magic": ["сиқыр", "пері", "жезтырнақ", "жалмауыз", "айдаһар", "ғажайып"],
    "honesty": ["адал", "шындық", "әділ", "өтірікші"],
}



def get_avg_sentence_len(text):
    # split by punctuation
    sentences = re.split(r"[.!?…]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences:
        return 0
    # avg word count per sentence
    counts = [len(s.split()) for s in sentences]
    return np.mean(counts)

def get_theme(text):
    text_lower = text.lower()
    scores = {k: 0 for k in THEME_KEYWORDS}

    for theme, keywords in THEME_KEYWORDS.items():
        for k in keywords:
            if k in text_lower:
                scores[theme] += 1

    # Return the theme with highest matches, or default to folklore
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "folklore"

def get_length_bucket(count):
    if count < 250: return "very_short"
    if count < 500: return "short"
    if count < 800: return "medium"
    return "long"




# 1. stats for dynamic age grouping
all_lens = []
for story in stories:
    txt = story.get("story", "")
    avg_len = get_avg_sentence_len(txt)
    story["avg_sent_len"] = avg_len
    # ensure we have word count
    if not story.get("length"):
        story["length"] = len(txt.split())
    all_lens.append(avg_len)

# 2. Determine age boundaries (Quantiles)
q25, q50, q75 = np.quantile(all_lens, [0.25, 0.50, 0.75])
print(f"Age boundaries (avg words/sent): <{q25:.1f}, <{q50:.1f}, <{q75:.1f}")

def get_age_group(avg_len):
    if avg_len <= q25: return "4-5"
    if avg_len <= q50: return "6-7"
    if avg_len <= q75: return "8-9"
    return "10-12"

# 3. Label data and write JSONL
labeled_data = []

with open(output_train, "w", encoding="utf-8") as f_out:
    for story in stories:
        text = story.get("story", "").strip()
        if len(text) < 20: continue

        # Generate labels
        age_group = get_age_group(story["avg_sent_len"])
        theme = get_theme(text)
        len_bucket = get_length_bucket(story["length"])
        seed = story.get("seed", "").strip().replace("\n", " ")

        # Update story dict
        story["age_group"] = age_group
        story["theme"] = theme

        # Create control tag string
        control_tags = f"<AGE={age_group}> <THEME={theme}> <LENGTH={len_bucket}> <SEED={seed}>"
        story["control_tags"] = control_tags

        labeled_data.append(story)


        # Prompt: Tags + Title -> Story
        title = story.get("title_kaz", story.get("title", "")).replace("ертегісі", "").strip()
        user_prompt = f"{control_tags}\nҚазақша ертегі жаз: {title}."

        entry = {
            "messages": [
                {
                    "role": "system",
                    "content": "Сен қазақша ертегі жазатын тіл моделісің. Берілген параметрлерге (жас ерекшелігі, тақырып, ұзындық) сәйкес, балаларға арналған мейірімді, әдеби стильді сақта."
                },
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": text}
            ]
        }
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")


with open(output_json, "w", encoding="utf-8") as f:
    json.dump(labeled_data, f, indent=4, ensure_ascii=False)

print(f"Done! Labeled {len(labeled_data)} stories.")
print(f"Training data saved to: {output_train}")

Loaded 291 stories.
Age boundaries (avg words/sent): <8.3, <9.4, <11.3
Done! Labeled 290 stories.
Training data saved to: /content/train_with_tags.jsonl


In [ ]:
import re

pattern = re.compile(r"[ ,\.!?]+")

sorter = []

for v in doc:
  sorter.append({'content': v, 'num_sentence_aprox': sum(1 for _ in pattern.finditer(v["story"]))})

print(sorter)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from collections.abc import MutableMapping
import pandas as pd

def flatten_dict(d: MutableMapping, sep: str= '.') -> MutableMapping:
    flat_dict = pd.json_normalize(d, sep=sep).to_dict(orient='records')
    return flat_dict

df_doc = pd.DataFrame(flatten_dict(doc))
df_doc

,id,title,url,age_group,theme,length,seed,planner,story,personalization.child_name,personalization.parent_role,personalization.location
0,Altynsarin_0000,ізбасты,https://wikisource.org/wiki/%D0%86%D0%B7%D0%B1...,None,None,102,"Ыбырай Алтынсарин\nҚыпшақ Ізбасты би, он жасар...",None,"Ыбырай Алтынсарин\nҚыпшақ Ізбасты би, он жасар...",None,None,None
1,Altynsarin_0001,абылай хан,https://wikisource.org/wiki/%D0%90%D0%B1%D1%8B...,None,None,64,Ыбырай Алтынсарин\nКерей Қожаберген жырауға Аб...,None,Ыбырай Алтынсарин\nКерей Қожаберген жырауға Аб...,None,None,None
2,Altynsarin_0002,"айуанның естесі көп, бірақ адамдай толық ақылы...",https://wikisource.org/wiki/%D0%90%D0%B9%D1%83...,None,None,170,Ыбырай Алтынсарин\nИт адамға шын дос жануар.,None,Ыбырай Алтынсарин\nИт адамға шын дос жануар. Д...,None,None,None
3,Altynsarin_0003,алтын шеттеуік,https://wikisource.org/wiki/%D0%90%D0%BB%D1%82...,None,None,150,"Ыбырай Алтынсарин\nБір үлкен мейрам алдында, ә...",None,"Ыбырай Алтынсарин\nБір үлкен мейрам алдында, ә...",None,None,None
4,Altynsarin_0004,асыл шөп,https://wikisource.org/wiki/%D0%90%D1%81%D1%8B...,None,None,94,Ыбырай Алтынсарин\nЗылиха мен Бәтима деген бір...,None,Ыбырай Алтынсарин\nЗылиха мен Бәтима деген бір...,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
1098,ertegiler_0204,baqa men kunnin dostygy,https://ertegiler.kz/story/baqa-men-kunnin-dos...,None,None,406,"Ертеде, жасыл орманда, кішкентайбақаларөмір сү...",None,"Ертеде, жасыл орманда, кішкентайбақаларөмір сү...",None,None,None
1099,ertegiler_0205,zalmauyz kempirdin qateri,https://ertegiler.kz/story/zalmauyz-kempirdin-...,None,None,501,"Ертеде, кең далада, бір ауыл болыпты.",None,"Ертеде, кең далада, бір ауыл болыпты. Ауылдың ...",None,None,None
1100,ertegiler_0206,qustar nege soilei almaidy,https://ertegiler.kz/story/qustar-nege-soilei-...,None,None,388,Ерте заманда адамдар мен құстардың достығы мық...,None,Ерте заманда адамдар мен құстардың достығы мық...,None,None,None
1101,ertegiler_0207,dostarmen otkizgen zhazgy demalys,https://ertegiler.kz/story/dostarmen-otkizgen-...,None,None,130,"Жазғы демалыс басталғанда, мен достарыммен бір...",None,"Жазғы демалыс басталғанда, мен достарыммен бір...",None,None,None


In [ ]:
import json
file_path = "/content/manually_checked_plan.json"
with open(file_path, "r", encoding="utf-8") as json_file:
    manual = json.load(json_file)
print(len(manual))

45


In [ ]:
from collections.abc import MutableMapping
import pandas as pd

def flatten_dict(d: MutableMapping, sep: str= '.') -> MutableMapping:
    flat_dict = pd.json_normalize(d, sep=sep).to_dict(orient='records')
    return flat_dict

df_manual = pd.DataFrame(flatten_dict(manual))
df_manual

,url,age_group,theme,story,planner.setting,planner.characters,planner.plot.beginning,planner.plot.conflict,planner.plot.development,planner.plot.climax,planner.plot.resolution,planner.plot.moral,title,themes
0,https://balalaralemi.kz/article/2094/Qu-tulki,2-5,"аңдар, айла, қулық",Жолбарыс бір түлкіні ұстап алады. Түлкі құйрығ...,"Орман, құстың ұясы мен түлкінің жорық жерлері","[{'name': 'Түлкі', 'role': 'айлакер, қу кейіпк...",Жолбарыс бір түлкіні ұстап алады.,Түлкі жолбарысқа: Мені жей алмайсың! Мен осы о...,"[Жолбарыс түлкіні жемекші болады., Түлкі түрлі...",Аңдар барлығы түлкінің артындағы жолбарыстан к...,Жолбарыс аңдар шыныменде түлкіден қашып жатыр ...,Қиын сәттерде амалсыздан қулыққа бару өміріңді...,NaN,NaN
1,https://balalaralemi.kz/article/1716/Ai-men-kun,2-5,"сұлу қыздар, ұрсысу, ренжіту",Өткен заманда біреудің Айсұлу және Күнсұлу дег...,"Аспан әлемі, күн мен айдың мекені, сұлу екі қы...","[{'name': 'Айсұлу', 'role': 'жылы, жұмсақ кейі...","Ай мен Күн аспанда бірге өмір сүріп, әлемге жа...",Екеуі ойнап отырғанда ұрсысып қалыпты.,[Айсұлу Күнсұлуға саған қарағанда мен сұлумын ...,Айсұлу бетінде Күнсұлу тырнап алған тыртық қал...,"Айсұлу бетіндегі тыртығынан, Күнсұлу Айсұлуға ...",Болмайтын нәрсеге көп ұрсысуға болмайды.,NaN,NaN
2,https://balalaralemi.kz/article/1128/ZHalqau-q...,2-7,"еріншектік, еңбектену, аңдар",Баяғы өткен заманда орманда құрғақшылық болыпт...,"Орман, аңдар барлығы еңбектеніп құдық қазғалы ...","[{'name': 'Жалқау қоян', 'role': 'негізгі кейі...",Орманда қуаңшылық болып аңдар құдық қазуға шеш...,Қоян мен кішкентаймын ғой деп құдық қазуға көм...,"[Аңдар қоянға су бермейміз деп шешеді., Қоянны...",Қоян мен піл бірге құдықты қайта қазады.,Қоян да құдықтан су іше алатын болады.,Еңбек — өмірдің негізі.,NaN,NaN
3,https://balalaralemi.kz/article/587/Tazsha-bala,6-10,"айлакерлік, тапқырлық",Ерте заманда бір тазша баланың әке-шешесі өліп...,"Хан сарайы, ауыл, дәстүрлі орта","[{'name': 'Тазша бала', 'role': 'негізгі кейіп...",Тазша бала әкесін Ханның артық салығынан құтқа...,Тазша бала ханның қиын сұрақтарына қарсы тұруы...,"[Тазша бала түрлі тапсырмалардан өтеді., Хан о...",Тазша бала ханның ең қиын сынағынан өтеді.,"Хан оны мақтап, қызын береді.",Ақыл мен тапқырлық — үлкен күш.,NaN,NaN
4,"https://balalaralemi.kz/article/154/Ur,toqpaq!",4-8,"сиқыр, кедей, шал мен кемпір, балалар",Бұрынғы өткен заманда бір шал мен кемпір болып...,Бұрынғы өткен заманда,"[{'name': 'Кедей шал мен кемпір', 'role': 'нег...",Қаз кедей шалдың торына түсіп қалады.,Қаздан шал қалаған затын сұрайды.,[Кедей шалдың әр алған затын балалар алып қояд...,Балалар шалға піс қазан мен құс есекті ұр тоқп...,Кедей адам бақытқа жетеді.,Адалдық — ең үлкен байлық.,NaN,NaN
5,https://balalaralemi.kz/article/151/Qarttyn-ul...,7-12,атаның өсиеті,Бір бай өлер уақытында ұлын шақырып алып:\n- Ұ...,Байдың үйі,"[{'name': 'Қарт әке', 'role': 'дана кейіпкер',...",Қарт әке ұлына өмірлік ақылын бергісі келеді.,Ұл кей ақылдарды бірден түсінбейді.,"[Атасы өлер алдында өсиетін айтып кетеді., Ұлы...",Ұл қиындықта әке өсиетін дұрыс қолданады.,Ол толыққанды азамат болудың мәнін түсінеді.,Даналық — атадан балаға берілетін қазына.,NaN,NaN
6,https://balalaralemi.kz/article/5997/Akyldy-qoyan,4-7,"салық, зұлым патшаның көзін құрту","Ерте ерте ертеде, Ешкі жүні бөртеде, өте бір б...","Ерте ерте ертеде, Ешкі жүні бөртеде, өте бір б...","[{'name': 'Ақылды қоян', 'role': 'негізгі кейі...","Орманда барлығы еңбекқор, пайда тапқыш.","Аңдар арасында талас болып, арыстан мәселені ө...","[Қоян ең соңғы болып салық төлеуге барады., Ар...","Қоян ең соңғы айласын қолданып, жыртқышты адас...",Қоян арыстанды қамап тастайды.,Ақыл — әлсіздің қорғанышы.,NaN,NaN
7,https://balalaralemi.kz/article/5950/Ush-ayu,4-7,әлсіздерге күш көрсету,Ертеде қалың тоғай ішіндегі үлкен бір үңгірде ...,"Үш аюдың үйі, орман","[{'name': 'Ақ төс аю', 'role': 'негізгі кейіпк...",Ең ірісі және мықтысы ақ төс аю болыпты.,"Күштілігіне мақтанып, кішкене аңдарға маза бер...","[Қасқыр мен түлкіге әлімжеттік көрсетті., Бір

In [ ]:
df_out = df_doc[~(df_doc['url'].isin(df_manual['url']))]
df_out

,id,title,url,age_group,theme,length,seed,planner,story,personalization.child_name,personalization.parent_role,personalization.location
0,Altynsarin_0000,ізбасты,https://wikisource.org/wiki/%D0%86%D0%B7%D0%B1...,None,None,102,"Ыбырай Алтынсарин\nҚыпшақ Ізбасты би, он жасар...",None,"Ыбырай Алтынсарин\nҚыпшақ Ізбасты би, он жасар...",None,None,None
1,Altynsarin_0001,абылай хан,https://wikisource.org/wiki/%D0%90%D0%B1%D1%8B...,None,None,64,Ыбырай Алтынсарин\nКерей Қожаберген жырауға Аб...,None,Ыбырай Алтынсарин\nКерей Қожаберген жырауға Аб...,None,None,None
2,Altynsarin_0002,"айуанның естесі көп, бірақ адамдай толық ақылы...",https://wikisource.org/wiki/%D0%90%D0%B9%D1%83...,None,None,170,Ыбырай Алтынсарин\nИт адамға шын дос жануар.,None,Ыбырай Алтынсарин\nИт адамға шын дос жануар. Д...,None,None,None
3,Altynsarin_0003,алтын шеттеуік,https://wikisource.org/wiki/%D0%90%D0%BB%D1%82...,None,None,150,"Ыбырай Алтынсарин\nБір үлкен мейрам алдында, ә...",None,"Ыбырай Алтынсарин\nБір үлкен мейрам алдында, ә...",None,None,None
4,Altynsarin_0004,асыл шөп,https://wikisource.org/wiki/%D0%90%D1%81%D1%8B...,None,None,94,Ыбырай Алтынсарин\nЗылиха мен Бәтима деген бір...,None,Ыбырай Алтынсарин\nЗылиха мен Бәтима деген бір...,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
1098,ertegiler_0204,baqa men kunnin dostygy,https://ertegiler.kz/story/baqa-men-kunnin-dos...,None,None,406,"Ертеде, жасыл орманда, кішкентайбақаларөмір сү...",None,"Ертеде, жасыл орманда, кішкентайбақаларөмір сү...",None,None,None
1099,ertegiler_0205,zalmauyz kempirdin qateri,https://ertegiler.kz/story/zalmauyz-kempirdin-...,None,None,501,"Ертеде, кең далада, бір ауыл болыпты.",None,"Ертеде, кең далада, бір ауыл болыпты. Ауылдың ...",None,None,None
1100,ertegiler_0206,qustar nege soilei almaidy,https://ertegiler.kz/story/qustar-nege-soilei-...,None,None,388,Ерте заманда адамдар мен құстардың достығы мық...,None,Ерте заманда адамдар мен құстардың достығы мық...,None,None,None
1101,ertegiler_0207,dostarmen otkizgen zhazgy demalys,https://ertegiler.kz/story/dostarmen-otkizgen-...,None,None,130,"Жазғы демалыс басталғанда, мен достарыммен бір...",None,"Жазғы демалыс басталғанда, мен достарыммен бір...",None,None,None


In [ ]:
import json

file_path = "/content/removed_manual.json"
with open(file_path, "w") as json_file:
  json.dump(df_out.to_dict('records'), json_file, indent=4)

## Dataset for instruction tuning without planning

In [ ]:
import json
import random

with open('/content/scraped_stories (1).json', 'r', encoding='utf-8') as f:
    stories = json.load(f)

prompt_templates = [
    "Осы тақырып {title} бойынша ертегі құрастыр.",
    "Берілген тақырып бойынша қазақша ертегі жаз: {title}.",
    "Осы тақырыпта {title} қазақша ертегі жаз. Басталуы: {seed}",
    "Моральмен аяқталатын қазақша ертегі жаз: Тақырыбы {title}.",
    "Қазақ ауыз әдебиеті стилінде ертегі жаз: Тақырыбы {title}.",
    "Қазақ фольклоры стилінде ертегі жаз, шамамен {length} сөз.",
    "Осы тақырыпқа толық қазақша ертегіні баяндап бер: {title}.",
    "Берілген сойлем бойынша ертегіні жалғастыр: {seed}"
]

with open('train.jsonl', 'w', encoding='utf-8') as f:
    for story in stories:
        title = story.get('title_kaz', '').replace("ертегісі","").strip()
        seed = story.get('seed', '').strip()
        length = story.get('length', 100)
        output_text = story.get('story', '').strip()

        if not output_text:
            continue

        usable_templates = []
        for t in prompt_templates:
            if "{seed}" in t and not seed:
                continue
            usable_templates.append(t)

        if not usable_templates:
            usable_templates = [
                "Қазақша ертегі жаз: {title}.",
                "Ертегіні толық баяндап бер: {title}."
            ]

        for _ in range(4):
            template = random.choice(usable_templates)
            user_prompt = template.format(title=title, seed=seed, length=length)

            entry = {
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "Сен қазақша ертегі жазатын тіл моделіcің. "
                            "Балаларға арналған мейірімді, әдеби стильді сақта, "
                            "дәстүрлі қазақ ертегілерінің құрылымын ұстан."
                        )
                    },
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": output_text}
                ]
            }

            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print("✔ Датасет дайын: train.jsonl")
